## **Modelación del Modelo y Selección de Variables**

* En este capítulo se aborda el proceso de modelación del precio de los vehículos, así como la selección de las variables más relevantes para el desempeño del modelo. Si bien en etapas previas del análisis exploratorio se identificó que variables como el año del vehículo y el kilometraje presentan una fuerte influencia sobre el precio, en esta fase se evalúa su impacto dentro del modelo predictivo, junto con otras variables categóricas y numéricas.

* Es importante destacar que, en el mercado automotriz, los vehículos nuevos experimentan una depreciación significativa, cercana al 40%, incluso cuando han recorrido pocos kilómetros fuera de la agencia. Este comportamiento justifica la inclusión del año y del kilometraje como variables clave, ya que capturan tanto el efecto del paso del tiempo como el desgaste del vehículo.

* Durante el proceso de modelación, se analizó el comportamiento del modelo ante distintas combinaciones de variables, evaluando su capacidad de generalización mediante métricas como el coeficiente de determinación ($R^2$) y validación cruzada. Asimismo, se examinó la presencia de sobreajuste (overfitting) y subajuste (underfitting), comparando el desempeño del modelo en los conjuntos de entrenamiento, validación y prueba.

* A partir de este análisis, se tomaron decisiones informadas sobre la inclusión o exclusión de variables, buscando un equilibrio entre la complejidad del modelo y su capacidad de generalización, con el objetivo de construir un modelo robusto y consistente con la dinámica real del mercado automotriz.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [2]:
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('/content/drive/MyDrive/dataset/car_sales_data_clean.csv')

In [4]:
df.head()

,Fabricante,Modelo,Motor,Combustible,Año,Kilometraje,Precio
0,Ford,Fiesta,1.0,Petrol,2002,127300,3074
1,Porsche,718 Cayman,4.0,Petrol,2016,57850,49704
2,Ford,Mondeo,1.6,Diesel,2014,39190,24072
3,VW,Polo,1.0,Petrol,2006,127869,4101
4,Ford,Focus,1.4,Petrol,2018,33603,29204


##

## **Partición del conjunto de datos**

In [5]:
X = df.drop(['Precio'], axis=1)
y = df['Precio']

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train,X_val,y_train,y_val = train_test_split(X_train,y_train,test_size=0.25,random_state=42)

* El conjunto de datos se dividió en tres subconjuntos: **entrenamiento**, **validación** y **prueba**. Inicialmente, se reservó el 20% de los datos para el conjunto de prueba, el cual se utilizó exclusivamente para evaluar el desempeño final del modelo. Posteriormente, el 80% restante se dividió en un 75% para entrenamiento y un 25% para validación, resultando en una partición final de 60% entrenamiento, 20% validación y 20% prueba.

## **Regresión lineal múltiple**

* La **regresión lineal múltiple** es un modelo estadístico que permite estimar el valor de una variable dependiente a partir de dos o más variables independientes. A diferencia de la regresión lineal simple, este modelo considera que el fenómeno a predecir está influenciado por varios factores al mismo tiempo.

* En el contexto automotriz, el precio de un vehículo no depende de una sola característica, sino de una combinación de factores como el fabricante, el modelo, el tipo de motor y el año de fabricación.

* El modelo busca encontrar los valores de los coeficientes **$β$** que minimizan la suma de los errores entre los valores reales y los valores predichos. Para ello, se utiliza el método de mínimos cuadrados ordinarios (OLS), el cual ajusta una recta (o hiperplano) que se aproxima lo mejor posible a los datos observados.



In [8]:
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [9]:
num_features = [ 'Motor','Año','Kilometraje']
cat_features = ['Fabricante', 'Modelo','Combustible']


preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ]
)

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

In [11]:
lm = TransformedTargetRegressor(
    regressor=Pipeline(steps=[
        ('preprocess', preprocessor),
        ('model', LinearRegression())
    ]),
    func=np.log,
    inverse_func=np.exp
)

In [12]:
lm.fit(X_train, y_train)

TransformedTargetRegressor(func=<ufunc 'log'>, inverse_func=<ufunc 'exp'>,
                           regressor=Pipeline(steps=[('preprocess',
                                                      ColumnTransformer(transformers=[('num',
                                                                                       StandardScaler(),
                                                                                       ['Motor',
                                                                                        'Año',
                                                                                        'Kilometraje']),
                                                                                      ('cat',
                                                                                       OneHotEncoder(handle_unknown='ignore'),
                                                                                       ['Fabricante',
                                                                                        'Modelo',
                                                                                        'Combustible'])])),
                                                     ('model',
                                                      LinearRegression())]))

La función **TransformedTargetRegressor** permite aplicar una transformación matemática directamente sobre la variable objetivo (target) sin necesidad de modificar manualmente los datos originales.


* La función func = np.log transforma el precio a escala logarítmica durante el entrenamiento.

* El modelo (en este caso una Regresión Lineal) aprende sobre esta escala transformada, lo que:

* Reduce la asimetría del target

* Disminuye el impacto de valores extremos

* Linealiza relaciones no lineales (como la depreciación de los autos)

Posteriormente:

La función inverse_func = np.exp revierte automáticamente la predicción a la escala original del precio (dólares).

Esto permite:

* Entrenar el modelo en una escala más estable

* Evaluar e interpretar resultados en unidades reales

* En otras palabras, el flujo es:

* Precio original → log(precio)

* Entrenamiento del modelo en la escala logarítmica

* Predicción → exp(predicción) → precio real

## **Métricas**

### **Coeficiente de determinación**

In [13]:
lm.score(X_train,y_train)

0.9912428113757339

In [14]:
lm.score(X_test,y_test)

0.9912708321898883

In [15]:
lm.score(X_val,y_val)

0.991227501955559

### **Error absoluto medio**

In [16]:
y_train_pred = lm.predict(X_train)
y_test_pred = lm.predict(X_test)
y_val_pred = lm.predict(X_val)

In [17]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score

In [18]:
print('MAE train: ',mean_absolute_error(y_train,y_train_pred))
print('MAE test: ',mean_absolute_error(y_test,y_test_pred))
print('MAE val: ',mean_absolute_error(y_val,y_val_pred))

MAE train:  663.6717994070796
MAE test:  644.793674661757
MAE val:  674.9826096895467


| Modelo                        | MAE Train | MAE Test | MAE Val |
|------------------------------|-----------|----------|---------|
| Datos sin tratar             | 5,779.99  | 5,781.50 | 5,678.01 |
| Datos tratados               | 663.67    | 644.79   | 674.98  |


* El modelo entrenado con datos previamente limpiados y transformados presenta una mejora sustancial en el desempeño predictivo respecto al modelo entrenado con datos sin tratamiento. En particular, el error absoluto medio (MAE) se reduce de valores cercanos a las 5,800 unidades a aproximadamente 650 USD en los conjuntos de entrenamiento, prueba y validación.

* Esta diferencia evidencia que la calidad de los datos tiene un impacto determinante en la capacidad predictiva del modelo, incluso manteniendo el mismo algoritmo de regresión.

### **Validación cruzada**

In [19]:
cross_val_score(lm,X_train,y_train,cv=5)

array([0.99168808, 0.9908805 , 0.99167021, 0.98983844, 0.99198717])

In [20]:
cross_val_score(lm,X_test,y_test,cv=5)

array([0.98963534, 0.99298921, 0.99080213, 0.99217911, 0.99196208])

In [21]:
cross_val_score(lm,X_val,y_val,cv=5)

array([0.99399166, 0.99044398, 0.99011491, 0.99107334, 0.98849112])

* En el modelo de **regresión lineal múltiple**, se aplicaron técnicas de validación cruzada con el objetivo de evaluar su capacidad de generalización y detectar posibles problemas de sobreajuste (overfitting).

* Los datos fueron divididos en conjuntos de entrenamiento, prueba y validación, utilizando una validación cruzada de 5 particiones (5-fold cross-validation). Los resultados obtenidos muestran un desempeño consistente entre los distintos subconjuntos, sin caídas significativas en la métrica de desempeño, lo cual indica que el modelo no está memorizando los datos, sino que aprende patrones generalizables.

* La estabilidad del **$R²$** a lo largo de los distintos folds confirma la solidez del modelo y descarta la presencia de overfitting, incluso al evaluar datos no vistos durante el entrenamiento.

## **XGBoost**



In [22]:
from xgboost import XGBRegressor

* El algoritmo **XGBoost (Extreme Gradient Boosting)** es uno de los métodos más exitosos en competencias de Kaggle, debido a su alta capacidad predictiva y eficiencia computacional. Pertenece a la familia de los árboles de gradiente (Gradient Boosting Trees), los cuales construyen modelos fuertes a partir de la combinación de múltiples modelos débiles.

* XGBoost funciona mediante la creación secuencial de un conjunto de árboles de decisión, donde cada nuevo árbol se entrena para corregir los errores cometidos por los árboles anteriores. De esta manera, el modelo mejora progresivamente su desempeño a lo largo del entrenamiento.

In [26]:
num_features = [ 'Motor','Año','Kilometraje']
cat_features = ['Fabricante','Modelo','Combustible']


preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ],
    remainder='passthrough'
)


* Dado que **XGBoost** es un modelo basado en árboles de decisión, no requiere transformaciones de escala en las variables numéricas. Por ello, se utilizó un **ColumnTransformer** que aplica codificación únicamente a las variables categóricas, permitiendo que las variables numéricas pasen sin modificaciones.”

In [50]:
xgb = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('xgb', XGBRegressor(
        n_estimators=600,
        learning_rate=0.01,
        max_depth=4,
        random_state=42,
        device = "cuda"
    ))
])

El modelo **XGBRegressor** cuenta con diversos hiperparámetros que controlan su comportamiento y complejidad. Entre los más importantes se encuentran los siguientes:

🔹**n_estimators:** Este parámetro define el número total de árboles de decisión que se entrenan. Cada árbol contribuye de forma incremental a la predicción final del modelo. Un mayor número de árboles puede mejorar el desempeño, aunque también incrementa el riesgo de sobreajuste si no se controla adecuadamente.



🔹**learning_rate:**  La tasa de aprendizaje controla la magnitud del ajuste que realiza cada árbol nuevo sobre el modelo existente. Un valor bajo permite que el aprendizaje sea más gradual y reduce el riesgo de overfitting, aunque requiere un mayor número de árboles para alcanzar un buen desempeño. Cada árbol intenta corregir los errores cometidos por los árboles anteriores.



🔹 **max_depth:** Este parámetro indica la profundidad máxima de cada árbol. Árboles demasiado profundos pueden capturar ruido y provocar sobreajuste, mientras que árboles poco profundos pueden no capturar correctamente la complejidad del problema. Por ello, es importante encontrar un equilibrio adecuado.



🔹**random_state:** El parámetro random_state fija el estado aleatorio del modelo, lo que garantiza la reproducibilidad de los resultados. De esta forma, al ejecutar el mismo código con los mismos datos, se obtienen resultados consistentes.



In [51]:
xgb.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Fabricante', 'Modelo',
                                                   'Combustible'])])),
                ('xgb',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=None, device='cuda',
                              early_stopping_rounds=None,
                              enab...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.01,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=4, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=600, n_jobs=None,
                              num_parallel_tree=None, ...))])

* Se integró el modelo XGBoost dentro de un Pipeline, lo que permite un flujo de trabajo más ordenado y reproducible al combinar el preprocesamiento de datos y el entrenamiento del modelo en una sola estructura.

* Aunque XGBoost es un modelo basado en árboles de decisión y no requiere transformaciones de escala en las variables numéricas, sí es necesario realizar un tratamiento adecuado de las variables categóricas. Para ello, estas variables se convirtieron a formato One-Hot Encoding, lo que permite que el modelo procese correctamente la información categórica.

## **Métricas**

### **Coeficiente de determinación**

In [52]:
xgb.score(X_train,y_train)

0.9923768043518066

In [53]:
xgb.score(X_test,y_test)

0.9913422465324402

In [54]:
xgb.score(X_val,y_val)

0.9916514158248901

### **Validación cruzada**

In [55]:
cross_val_score(xgb,X_train,y_train,cv=5)

array([0.99144965, 0.99128187, 0.99012691, 0.99097061, 0.991862  ])

In [56]:
cross_val_score(xgb,X_test,y_test,cv=5)

array([0.98609841, 0.988123  , 0.98936361, 0.98962015, 0.98870438])

In [57]:
cross_val_score(xgb,X_val,y_val,cv=5)

array([0.98892707, 0.98833144, 0.99031919, 0.98739946, 0.98736656])

### **Error absoluto medio**

In [58]:
y_train_pred = xgb.predict(X_train)
y_test_pred = xgb.predict(X_test)
y_val_pred = xgb.predict(X_val)

In [59]:
print('MAE train: ',mean_absolute_error(y_train,y_train_pred))
print('MAE test: ',mean_absolute_error(y_test,y_test_pred))
print('MAE val: ',mean_absolute_error(y_val,y_val_pred))

MAE train:  958.5062255859375
MAE test:  973.469970703125
MAE val:  991.5188598632812




Aunque el modelo basado en XGBoost presenta un alto valor de **$R²$**, su error absoluto medio es significativamente mayor en comparación con la regresión lineal múltiple. Esto indica que, si bien XGBoost explica una mayor proporción de la varianza, la regresión lineal ofrece predicciones más precisas en términos de error real, lo que la convierte en la mejor opción para este problema.”

## **Comparación de desempeño entre modelos**

| Modelo | Estado de los datos | MAE Train | MAE Test | MAE Validación |
|------|-------------------|-----------|----------|----------------|
| Regresión lineal múltiple | Datos sucios (sin limpieza ni transformación) | 5,779.99 | 5,781.50 | 5,678.01 |
| Regresión lineal múltiple | Datos limpios + transformación logarítmica | 663.67 | 644.79 | 674.98 |
| XGBoost | Datos limpios | 950 | 973 | 991 |


In [60]:
def Predict(Fabricante,Modelo,Motor,Combustible,Año,Kilometraje):
  x = pd.DataFrame(
      {
          'Fabricante':[Fabricante],
          'Modelo':[Modelo],
          'Motor':[Motor],
          'Combustible':[Combustible],
          'Año':[Año],
          'Kilometraje':[Kilometraje]
      }
  )

  return lm.predict(x)

In [76]:
Predict('Toyota','RAV4',2.4,'Petrol',2016,150000)

array([16610.55274467])

In [77]:
import joblib

In [79]:
joblib.dump(lm,'/content/drive/MyDrive/modelo/model.pkl')

['/content/drive/MyDrive/modelo/model.pkl']

## **Conclusión**

* En este proyecto, el algoritmo de regresión lineal múltiple demostró un desempeño superior frente a XGBoost para la tarea de predicción del precio de vehículos. A pesar de que XGBoost es un modelo ampliamente reconocido por su alto rendimiento y por haber sido ganador en múltiples competencias de Kaggle, en este caso no logró superar a un modelo más simple en términos de error absoluto medio (MAE).

* La **regresión lineal múltiple** no solo presentó menores errores en entrenamiento, prueba y validación, sino que además mostró una mayor estabilidad y capacidad de generalización, evidenciando que el problema posee una estructura predominantemente log-lineal.

* Otro aspecto clave es la interpretabilidad. En entornos reales, especialmente cuando se trabaja con clientes o tomadores de decisiones, no solo se requiere un modelo preciso, sino también uno que pueda explicarse con claridad. La regresión lineal múltiple permite entender de forma directa cómo cada variable influye en el precio del vehículo, algo que resulta considerablemente más complejo en modelos como XGBoost.

* Asimismo, este análisis pone en evidencia la importancia de la **limpieza de datos** y del **preprocesamiento**. La eliminación de **valores atípicos** y la **transformación logarítmica** del precio tuvieron un impacto mucho mayor en el desempeño del modelo que el uso de algoritmos más complejos. Esto demuestra que modelos considerados “simples” pueden competir e incluso superar a algoritmos avanzados cuando los datos están correctamente tratados.

* En **conclusión**, este estudio refuerza la idea de que la calidad de los datos y la correcta formulación del problema son factores determinantes, y que la elección del modelo debe basarse no solo en su complejidad o popularidad, sino en su adecuación al problema, interpretabilidad y eficiencia computacional.
